# Lab 008 — Read the curves, fix the probabilities

**Lesson:** [`lessons/0008-metrics-calibration.html`](../lessons/0008-metrics-calibration.html) · **Phase / Year:** Phase 1 / Year 1

**Skill you are practising:** choose the right curve (ROC vs PR) and read it against the prevalence baseline, then check and fix probability calibration (reliability curve + Brier).

**Exit criteria:** the EXIT TICKET prints ROC-AUC vs PR-AUC (with prevalence), the raw-vs-calibrated Brier scores (with AUROC unchanged), and your one-sentence cause.

---

### How this notebook works
- **PROVIDED** cells are complete — just run them.
- **TODO** cells have blanks (`____`). Fill them in.
- **CHECK** cells auto-run and give you immediate feedback — don't edit them.
- Run top to bottom. When the **EXIT TICKET** prints cleanly, paste it back to your teacher (or say *"lab done"*).

### Environment
One-time setup from the repo root: `bash labs/setup-env.sh`.  
Then select kernel **Relational Labs (.venv)** or interpreter `.venv/bin/python`.

## Setup — PROVIDED (run me)

In [ ]:
# PROVIDED
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

RS = 0
print("setup ok")

## Data — PROVIDED

One imbalanced dataset (~18% positive) with a little label noise — enough to make ROC look rosier than PR *and* to leave a RandomForest mis-calibrated.

In [ ]:
# PROVIDED
X, y = make_classification(n_samples=12000, n_features=20, n_informative=6,
                           n_redundant=4, weights=[0.85, 0.15],
                           flip_y=0.08, class_sep=0.8, random_state=RS)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, stratify=y, random_state=RS)
prevalence = yte.mean()
print("prevalence (PR no-skill baseline):", round(prevalence, 4))

## Task 1 — ROC vs PR against the baseline — TODO

Fit a RandomForest, get its positive-class probabilities on the test set, and compute both
ROC-AUC and PR-AUC (average precision). Compare PR-AUC to the prevalence baseline.

Hint: `rf.predict_proba(Xte)[:, 1]`; `average_precision_score` is PR-AUC.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=RS).fit(Xtr, ytr)

# TODO: positive-class probabilities, then the two AUCs.
p_rf = ____
roc = ____
pr  = ____

print(f"ROC-AUC {roc:.3f}   PR-AUC {pr:.3f}   PR baseline (prevalence) {prevalence:.3f}")

In [ ]:
# CHECK — do not edit.
assert 0.5 < pr < roc, "Expected PR-AUC below ROC-AUC on this imbalanced set."
assert pr > prevalence + 0.1, "PR-AUC should beat the prevalence baseline by a clear margin."
print(f"Task 1 ok — ROC ({roc:.3f}) reads higher than PR ({pr:.3f}); PR is real skill above {prevalence:.3f}.")

## Task 2 — Read the reliability curve — TODO

Compute the calibration (reliability) curve of the raw RandomForest with 10 bins, then look at
a mid-range bin: is the model over- or under-confident?

Hint: `calibration_curve(yte, p_rf, n_bins=10, strategy="uniform")` returns
`(frac_pos, mean_pred)`.

In [ ]:
# TODO: compute the reliability curve.
frac_pos, mean_pred = ____

for mp, fp in zip(mean_pred, frac_pos):
    print(f"predicted {mp:.3f} -> observed {fp:.3f}")

In [ ]:
# CHECK — do not edit.
assert len(frac_pos) >= 5, "Expected several reliability bins."
gap = np.max(np.abs(np.array(frac_pos) - np.array(mean_pred)))
assert gap > 0.05, "A raw RandomForest should sit visibly off the diagonal somewhere."
print(f"Task 2 ok — largest predicted-vs-observed gap is {gap:.3f}: the curve is off the diagonal.")

## Task 3 — Calibrate and check Brier — TODO

Wrap a fresh RandomForest in `CalibratedClassifierCV` with `method="sigmoid"` and `cv=5`, fit on
train, and compare the Brier score (and AUROC) of raw vs calibrated on the test set.

In [ ]:
brier_raw = brier_score_loss(yte, p_rf)

# TODO: build and fit the calibrated classifier, then get its probabilities.
cal = ____
p_cal = ____

brier_cal = brier_score_loss(yte, p_cal)
roc_cal = roc_auc_score(yte, p_cal)
print(f"Brier raw {brier_raw:.4f} -> calibrated {brier_cal:.4f}")
print(f"AUROC raw {roc:.3f} -> calibrated {roc_cal:.3f} (ranking barely moves)")

In [ ]:
# CHECK — do not edit.
assert brier_cal <= brier_raw, "Calibration should not worsen the Brier score here."
assert abs(roc_cal - roc) < 0.03, "Calibration should leave ranking (AUROC) essentially unchanged."
print(f"Task 3 ok — Brier improved by {brier_raw - brier_cal:.4f} with AUROC ~unchanged.")

## Exit ticket — TODO

Fill the one-sentence takeaway, run, and paste the output back to your teacher.

In [ ]:
# TODO: complete the takeaway.
print("=== EXIT TICKET — Lesson 008 ===")
print(f"ROC-AUC {roc:.3f} vs PR-AUC {pr:.3f}  (PR baseline / prevalence {prevalence:.3f})")
print(f"Brier raw {brier_raw:.4f} -> calibrated {brier_cal:.4f}  (AUROC {roc:.3f} -> {roc_cal:.3f})")
print()
print("takeaway:", "____")   # why does ROC read higher, and what did calibration change (and not change)?